In [1]:
from pathlib import Path

import pandas as pd

# Resolve the repo root by walking up until the data folder appears, so the
# notebook runs regardless of the directory Jupyter is launched from.
BASE_DIR = Path.cwd()
while not (BASE_DIR / "data").is_dir() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "cleaned" / "Cambridge data_cleaned.csv"
OUTPUT_PATH = BASE_DIR / "data" / "cleaned" / "Cambridge data_diversity.csv"

# The cleaned data stores the social columns with R-style dotted names; map them
# to the readable names used across the repo's other scripts.
COLUMN_RENAME = {
    "Minors.0.18": "Minors 0-18",
    "Adults.18.60": "Adults 18-60",
    "Elders..60": "Elders >60",
    "X0.5km": "0-5km",
    "X5.30km": "5-30km",
    "X.30km": ">30km",
    "Work.from.Home..0km.": "Work from Home (0km)",
    "Living.Offshore": "Living Offshore",
    "Other.Qualification": "Other Qualification",
    "City.Public": "City Public",
    "Other.Method": "Other Method",
}

df = pd.read_csv(DATA_PATH).rename(columns=COLUMN_RENAME)

In [2]:
df.head()

,msoa21,start_date,sold_date,property_type,price_sold,num_bed_,num_bath,num_reception,Asian,Black,...,City Public,Rail,Cycle,Driving,Foot,Other Method,LAT,LONG,dist_london,dist_cambridge
0,E02003719,2019-05-26,2019-07-06,End terrace house,250000.0,1,1,1,10.7,3.2,...,6.7,0.4,19.6,32.6,7.4,0.8,52.22917,0.134733,82.112282,2.791735
1,E02003719,2018-04-05,2019-02-07,Maisonette,220000.0,2,1,0,10.7,3.2,...,6.7,0.4,19.6,32.6,7.4,0.8,52.22917,0.134733,82.112282,2.791735
2,E02003719,2018-10-25,2019-06-06,Terraced house,205000.0,3,1,1,10.7,3.2,...,6.7,0.4,19.6,32.6,7.4,0.8,52.22917,0.134733,82.112282,2.791735
3,E02003719,2019-08-28,2020-01-10,Terraced house,285000.0,2,1,1,10.7,3.2,...,6.7,0.4,19.6,32.6,7.4,0.8,52.22917,0.134733,82.112282,2.791735
4,E02003719,2021-12-09,2022-01-05,Semi-detached house,595000.0,4,2,2,10.7,3.2,...,6.7,0.4,19.6,32.6,7.4,0.8,52.22917,0.134733,82.112282,2.791735


**Shannon entropy**

$$
H = -\sum_{i=1}^K p_i \log(p_i)
$$

If all groups are equally represented, $p_i = \frac{1}{K}$, $H = -K \cdot \frac{1}{K} \log \frac{1}{K} = \log K$.

So people often use a normalized version
$$
H = \frac{-\sum_{i=1}^K p_i \log(p_i)}{\log K}
$$
which gives a value between 0 and 1
- $H=0$: no diversity
- $H=1$: max diversity

**Simpson diversity index / Gini-Simpson index**

Concentration measure:
$$
\lambda = \sum_{i=1}^K p_i^2
$$
Higher when one group dominates.

Diversity measure:
$$
D = 1 - \sum_{i=1}^K p_i^2
$$

Interpretation: the probablity that two randomly selected people from the region belong to different groups (higher when diverse)

maximum value: $1 - \frac{1}{K}$

normalized version:
$$
D = \frac{1 - \sum_{i=1}^K p_i^2}{1 - \frac{1}{K}}
$$

Normalisation refers to normalizing across number of classes

# Shannon

In [3]:
# Social blocks, named to match the cleaned data. The travel-method block has 6
# categories: the work-from-home travel method was dropped upstream because it
# duplicated the 0km distance-to-work column ("Work from Home (0km)").
ethnicity_cols = ["Asian", "Black", "Mixed", "White", "Other"]
age_cols = ["Minors 0-18", "Adults 18-60", "Elders >60"]
edu_cols = ["Low", "Medium", "High", "Other Qualification"]
travel_time_cols = ["Work from Home (0km)", "0-5km", "5-30km", ">30km", "Living Offshore"]
travel_method_cols = ["City Public", "Rail", "Cycle", "Driving", "Foot", "Other Method"]


def to_proportions(block):
    # Convert a block of percentage columns into row-wise proportions that sum
    # to 1, so each block is a valid probability distribution even when its
    # categories do not add up to exactly 100 (rounding, or a dropped category).
    return block.div(block.sum(axis=1), axis=0)

In [4]:
import numpy as np

def shannon(p):
    K = p.shape[1]
    return (
            -np.where(
            p>0,
            p * np.log(p),
            0
        ).sum(axis=1)
    ) / np.log(K)


In [5]:
df["ethnicity_shannon"] = shannon(to_proportions(df[ethnicity_cols]))
df["age_shannon"] = shannon(to_proportions(df[age_cols]))
df["edu_shannon"] = shannon(to_proportions(df[edu_cols]))
df["travel_time_shannon"] = shannon(to_proportions(df[travel_time_cols]))
df["travel_method_shannon"] = shannon(to_proportions(df[travel_method_cols]))

# Simpson

In [6]:
def simpson(p):
    K = p.shape[1]
    return (1 - (p ** 2).sum(axis=1)) / (1 - 1/K)

In [7]:
df["ethnicity_simpson"] = simpson(to_proportions(df[ethnicity_cols]))
df["age_simpson"] = simpson(to_proportions(df[age_cols]))
df["edu_simpson"] = simpson(to_proportions(df[edu_cols]))
df["travel_time_simpson"] = simpson(to_proportions(df[travel_time_cols]))
df["travel_method_simpson"] = simpson(to_proportions(df[travel_method_cols]))

In [8]:
df.head()

,msoa21,start_date,sold_date,property_type,price_sold,num_bed_,num_bath,num_reception,Asian,Black,...,ethnicity_shannon,age_shannon,edu_shannon,travel_time_shannon,travel_method_shannon,ethnicity_simpson,age_simpson,edu_simpson,travel_time_simpson,travel_method_simpson
0,E02003719,2019-05-26,2019-07-06,End terrace house,250000.0,1,1,1,10.7,3.2,...,0.465183,0.826462,0.824782,0.887363,0.70611,0.439358,0.795708,0.866853,0.913455,0.792462
1,E02003719,2018-04-05,2019-02-07,Maisonette,220000.0,2,1,0,10.7,3.2,...,0.465183,0.826462,0.824782,0.887363,0.70611,0.439358,0.795708,0.866853,0.913455,0.792462
2,E02003719,2018-10-25,2019-06-06,Terraced house,205000.0,3,1,1,10.7,3.2,...,0.465183,0.826462,0.824782,0.887363,0.70611,0.439358,0.795708,0.866853,0.913455,0.792462
3,E02003719,2019-08-28,2020-01-10,Terraced house,285000.0,2,1,1,10.7,3.2,...,0.465183,0.826462,0.824782,0.887363,0.70611,0.439358,0.795708,0.866853,0.913455,0.792462
4,E02003719,2021-12-09,2022-01-05,Semi-detached house,595000.0,4,2,2,10.7,3.2,...,0.465183,0.826462,0.824782,0.887363,0.70611,0.439358,0.795708,0.866853,0.913455,0.792462


In [9]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(df):,} rows to {OUTPUT_PATH}")

Wrote 59,787 rows to /home/user/graduate_modelling_camp/data/cleaned/Cambridge data_diversity.csv
